# Fine-Tuning T5 for SQuAD Question Answering

**Repository:** `finetuning-t5-question-answering`  
**Task:** UAS Deep Learning — Fine-Tuning HuggingFace Models, Task 2  
**Architecture Type:** Encoder-Decoder / Sequence-to-Sequence Transformer  
**Model:** `t5-base`  
**Dataset:** SQuAD  
**Problem Type:** Generative Question Answering

---

## 1. Learning Objectives

By the end of this notebook, you should be able to:

1. Build an end-to-end generative question answering pipeline using HuggingFace Transformers.
2. Load and inspect the SQuAD dataset.
3. Convert SQuAD examples into a T5 text-to-text format.
4. Tokenize context-question inputs and answer targets.
5. Fine-tune a T5 encoder-decoder model for question answering.
6. Evaluate the model using Exact Match and token-level F1-score.
7. Generate answers from unseen context-question pairs.

## 2. Theoretical Background

T5 is an encoder-decoder Transformer that treats many NLP problems as text-to-text tasks. For question answering, the input sequence can be formatted as a natural-language prompt containing the question and context, while the output sequence is the answer text.

Unlike extractive QA models that predict start and end positions in the context, T5 generates the answer token by token. This makes the task a sequence-to-sequence problem. The encoder reads the question and context, while the decoder generates the answer.

The end-to-end workflow is:

1. Load SQuAD dataset.
2. Inspect context, question, and answer fields.
3. Format each example as `question: ... context: ...`.
4. Tokenize inputs and target answers.
5. Fine-tune T5 using sequence-to-sequence training.
6. Generate predictions on validation/test examples.
7. Evaluate using Exact Match and F1-score.

## 3. Environment Setup

Run this notebook in Google Colab with GPU enabled. If the required libraries are not installed, uncomment and run the installation command.

In [1]:
# If running in Google Colab, uncomment the following line:
# !pip install -q transformers datasets accelerate sentencepiece scikit-learn matplotlib pandas numpy

## 4. Import Libraries and Set Configuration

This section imports the required libraries and defines global configuration values. The subset sizes are intentionally small to make the experiment practical on Google Colab T4 GPU.

In [2]:
import os
import re
import string
import random
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

RANDOM_STATE = 42
MODEL_NAME = "t5-base"
DATASET_NAME = "rajpurkar/squad"
OUTPUT_DIR = "./t5-squad-checkpoints"
FINAL_MODEL_DIR = "./t5-squad-final"

TRAIN_SIZE = 1000
VAL_SIZE = 200
TEST_SIZE = 200
MAX_INPUT_LENGTH = 384
MAX_TARGET_LENGTH = 48

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)

Model: t5-base
Dataset: rajpurkar/squad


## 5. Load SQuAD Dataset

The assignment specifies SQuAD as the dataset for Task 2. SQuAD contains context paragraphs, questions, and answer spans. In this notebook, the answer text is used as the generation target for T5.

In [3]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print("Available splits:", dataset.keys())
print("Training example keys:", dataset["train"][0].keys())
print("Example:")
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
Available splits: dict_keys(['train', 'validation'])
Training example keys: dict_keys(['id', 'title', 'context', 'question', 'answers'])
Example:
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly ap

## 6. Dataset Inspection

Before preprocessing, inspect source fields and answer structure. In SQuAD, the `answers` field contains a list of possible answer texts and their character start positions.

In [4]:
train_preview = pd.DataFrame(dataset["train"].select(range(5)))
print(train_preview[["id", "title", "question", "context", "answers"]])

for idx in range(2):
    row = dataset["train"][idx]
    print("\nExample", idx)
    print("Question:", row["question"])
    print("Context:", row["context"][:350], "...")
    print("Answer:", row["answers"]["text"][0])

                         id                     title  \
0  5733be284776f41900661182  University_of_Notre_Dame   
1  5733be284776f4190066117f  University_of_Notre_Dame   
2  5733be284776f41900661180  University_of_Notre_Dame   
3  5733be284776f41900661181  University_of_Notre_Dame   
4  5733be284776f4190066117e  University_of_Notre_Dame   

                                            question  \
0  To whom did the Virgin Mary allegedly appear i...   
1  What is in front of the Notre Dame Main Building?   
2  The Basilica of the Sacred heart at Notre Dame...   
3                  What is the Grotto at Notre Dame?   
4  What sits on top of the Main Building at Notre...   

                                             context  \
0  Architecturally, the school has a Catholic cha...   
1  Architecturally, the school has a Catholic cha...   
2  Architecturally, the school has a Catholic cha...   
3  Architecturally, the school has a Catholic cha...   
4  Architecturally, the school has a Cat

## 7. Create Lightweight Training, Validation, and Test Subsets

Fine-tuning T5-base can be heavier than DistilBERT. This notebook uses a practical subset. For stronger final results, increase the subset sizes or train on the full dataset if sufficient GPU time is available.

In [5]:
train_split = dataset["train"].shuffle(seed=RANDOM_STATE)
val_split = dataset["validation"].shuffle(seed=RANDOM_STATE)

small_train = train_split.select(range(min(TRAIN_SIZE, len(train_split))))
small_val = val_split.select(range(min(VAL_SIZE, len(val_split))))
small_test = val_split.select(range(min(VAL_SIZE, len(val_split)), min(VAL_SIZE + TEST_SIZE, len(val_split))))

print("Train size:", len(small_train))
print("Validation size:", len(small_val))
print("Test size:", len(small_test))

Train size: 1000
Validation size: 200
Test size: 200


## 8. Format SQuAD as a Text-to-Text Task

T5 expects text input and text output. The input is formatted as `question: ... context: ...`, and the target output is the first gold answer text.

In [6]:
def format_example(example):
    question = example["question"].strip()
    context = example["context"].strip()
    answer = example["answers"]["text"][0].strip() if len(example["answers"]["text"]) > 0 else ""
    return {
        "input_text": f"question: {question} context: {context}",
        "target_text": answer
    }

formatted_train = small_train.map(format_example)
formatted_val = small_val.map(format_example)
formatted_test = small_test.map(format_example)

print("Formatted input example:")
print(formatted_train[0]["input_text"][:500])
print("\nTarget answer:")
print(formatted_train[0]["target_text"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Formatted input example:
question: What percentage of Egyptians polled support death penalty for those leaving Islam? context: The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerate

Target answer:
84%


## 9. Tokenization

The tokenizer converts formatted input text and target answers into token IDs. Input truncation is applied to keep context length manageable, while target truncation is applied to answer sequences.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = formatted_train.map(preprocess_function, batched=True)
tokenized_val = formatted_val.map(preprocess_function, batched=True)
tokenized_test = formatted_test.map(preprocess_function, batched=True)

print(tokenized_train[0].keys())
print("Input token length:", len(tokenized_train[0]["input_ids"]))
print("Target token length:", len(tokenized_train[0]["labels"]))

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

dict_keys(['id', 'title', 'context', 'question', 'answers', 'input_text', 'target_text', 'input_ids', 'attention_mask', 'labels'])
Input token length: 164
Target token length: 3


## 10. Load T5 Model and Data Collator

`AutoModelForSeq2SeqLM` loads T5 for conditional generation. The data collator pads inputs and labels dynamically for efficient batching.

In [8]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

print("Model loaded successfully:", MODEL_NAME)

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully: t5-base


## 11. Define QA Evaluation Metrics

Exact Match checks whether the generated answer exactly matches the gold answer after normalization. F1-score measures token overlap between generated and reference answers, making it more flexible than exact string matching.

In [9]:
def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"(a|an|the)", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def f1_score_qa(prediction, ground_truth):
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    common = collections.Counter(prediction_tokens) & collections.Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if len(prediction_tokens) == 0 or len(ground_truth_tokens) == 0:
        return int(prediction_tokens == ground_truth_tokens)
    if num_same == 0:
        return 0
    precision = num_same / len(prediction_tokens)
    recall = num_same / len(ground_truth_tokens)
    return 2 * precision * recall / (precision + recall)


def evaluate_generated_answers(predictions, references):
    em_scores = [exact_match_score(p, r) for p, r in zip(predictions, references)]
    f1_scores = [f1_score_qa(p, r) for p, r in zip(predictions, references)]
    return {
        "exact_match": float(np.mean(em_scores)),
        "f1": float(np.mean(f1_scores))
    }

## 12. Training Configuration

This configuration is lightweight for Colab. T5-base can be memory-intensive, so the batch size is small. If computation becomes too slow, reduce `TRAIN_SIZE` or switch temporarily to `t5-small` for debugging only.

In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    report_to="none",
    seed=RANDOM_STATE
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Seq2SeqTrainer configured.")

Seq2SeqTrainer configured.


## 13. Fine-Tune the Model

This cell starts fine-tuning T5-base on the SQuAD subset. On Google Colab T4 GPU, this may take several minutes depending on runtime availability.

In [11]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss
1,0.332738,0.468789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.42676280975341796, metrics={'train_runtime': 110.1166, 'train_samples_per_second': 9.081, 'train_steps_per_second': 2.27, 'total_flos': 318299434905600.0, 'train_loss': 0.42676280975341796, 'epoch': 1.0})


## 14. Generate Answers on the Test Subset

After fine-tuning, the model generates answer text for held-out examples. The generated predictions are then compared with reference answers.

In [12]:
def generate_answer(question, context):
    input_text = f"question: {question.strip()} context: {context.strip()}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH).to(model.device)
    outputs = model.generate(**inputs, max_length=MAX_TARGET_LENGTH, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sample_predictions = []
sample_references = []

for example in small_test:
    pred = generate_answer(example["question"], example["context"])
    ref = example["answers"]["text"][0] if len(example["answers"]["text"]) > 0 else ""
    sample_predictions.append(pred)
    sample_references.append(ref)

print("Generated", len(sample_predictions), "predictions.")
for i in range(3):
    print("\nQuestion:", small_test[i]["question"])
    print("Prediction:", sample_predictions[i])
    print("Reference:", sample_references[i])

Generated 200 predictions.

Question: When did rapid warming begin and help vegetation?
Prediction: 13,000 BP
Reference: 13,000 BP

Question: How long did it take Johnson to respond to Kennedy?
Prediction: one week
Reference: approximately one week

Question: What measurement of time is used in polynomial time reduction?
Prediction: polynomial
Reference: polynomial time


## 15. Evaluate Generated Answers

Exact Match and F1 are computed over the test subset. Exact Match is strict, while F1 gives partial credit for overlapping answer tokens.

In [13]:
qa_metrics = evaluate_generated_answers(sample_predictions, sample_references)
print("QA Metrics:")
print(qa_metrics)

results_df = pd.DataFrame({
    "question": [ex["question"] for ex in small_test],
    "prediction": sample_predictions,
    "reference": sample_references
})
print(results_df.head())

QA Metrics:
{'exact_match': 0.59, 'f1': 0.7934209725739074}
                                            question       prediction  \
0  When did rapid warming begin and help vegetation?        13,000 BP   
1  How long did it take Johnson to respond to Ken...         one week   
2  What measurement of time is used in polynomial...       polynomial   
3  What is the receptor that killer T cells use t...  T cell receptor   
4  Along with poppet valve gears, what type of ge...          Corliss   

                reference  
0               13,000 BP  
1  approximately one week  
2         polynomial time  
3   T cell receptor (TCR)  
4                 Corliss  


## 16. Inference Demo

This section tests the fine-tuned model on one custom question-context pair.

In [14]:
custom_context = "The Eiffel Tower is a wrought-iron tower located in Paris, France. It was designed by Gustave Eiffel and completed in 1889."
custom_question = "Who designed the Eiffel Tower?"
custom_answer = generate_answer(custom_question, custom_context)
print("Question:", custom_question)
print("Generated answer:", custom_answer)

Question: Who designed the Eiffel Tower?
Generated answer: Gustave Eiffel


## 17. Final Conclusion

This notebook demonstrates an end-to-end HuggingFace fine-tuning pipeline for generative question answering using T5 and SQuAD. The pipeline includes dataset loading, format conversion into text-to-text input, tokenization, seq2seq training, answer generation, and QA metric evaluation.

After running the notebook, update this conclusion using the final Exact Match and F1-score reported in the evaluation section. In general, T5 is suitable for SQuAD-style question answering because its encoder-decoder structure can read question-context pairs and generate answer text directly.